<a href="https://colab.research.google.com/github/vilassn/whisper_android/blob/master/models_and_scripts/whisper_tflite_model_generation_and_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Install TensorFlow, Tranformers and datasets

### 环境配置

In [ ]:
!conda install -n asr python==3.9
!conda activate asr
!conda install -n asr ipykernel --update-deps --force-reinstall
!pip install transformers==4.42.1
!pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!pip install tensorflow==2.12.0, keras==2.12.0, numpy==1.23.5
!pip install "torchcodec==0.5.0"
!conda install conda-forge::ffmpeg==7.1.1
!pip install "datasets==4.0.0"
!pip install transformers==4.42.1 tensorflow==2.12.0 keras==2.12.0 numpy==1.23.5 #FINAL

## Configure model to be generated as per requirement

In [1]:
import requests
import json

######## Set the model as per requirement
model_name = "whisper-small"          # whisper-tiny, whisper-tiny.en, whisper-base, whisper-base.en, whisper-small, whisper-small.en

######## Set the language, task, and options as per requirement
language_code = "<|zh|>"             # <|en|>, <|fr|>, <|hi|>, <|ko|>, <|de|>, <|zh|>, <|ja|>, <|es|>, <|ar|>, <|ru|>, ...
task_code     = "<|transcribe|>"     # <|transcribe|>, <|translate|>
option_code   = "<|notimestamps|>"   # <|notimestamps|>, <|nocaptions|>

# URL of the JSON file which stores the code mappings
url = "https://huggingface.co/openai/whisper-large/resolve/main/added_tokens.json"

# Send a GET request to download the file
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Parse the JSON content
    code_mappings = response.json()
else:
    print(f"Failed to download the file. Status code: {response.status_code}")
    code_mappings = {}

# Construct forced_decoder_ids using the mappings
forced_decoder_ids = [
    [1, code_mappings[language_code]],
    [2, code_mappings[task_code]],
    [3, code_mappings[option_code]]
]

print(forced_decoder_ids)

[[1, 50260], [2, 50359], [3, 50363]]


##Import the libraries, load the model, do the inference

In [2]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from datasets import load_dataset
from transformers import WhisperProcessor, WhisperFeatureExtractor, TFWhisperForConditionalGeneration, WhisperTokenizer
#from transformers import TF

pretrained_model = f"openai/{model_name}"
tflite_model_path = f"{model_name}.tflite"
saved_model_dir = f"tf_{model_name}_saved"

feature_extractor = WhisperFeatureExtractor.from_pretrained(pretrained_model)
tokenizer = WhisperTokenizer.from_pretrained(pretrained_model, predict_timestamps=True)
processor = WhisperProcessor(feature_extractor, tokenizer)
model = TFWhisperForConditionalGeneration.from_pretrained(pretrained_model)

# Loading dataset
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
inputs = feature_extractor(ds[0]["audio"]["array"], sampling_rate=ds[0]["audio"]["sampling_rate"], return_tensors="tf")
input_features = inputs.input_features

# Generating Transcription
generated_ids = model.generate(input_features=input_features)
print(generated_ids)

transcription = processor.tokenizer.decode(generated_ids[0])
print(transcription)

# Save the model
# model.save(saved_model_dir) # not need to save here, saving using tf.saved_model.save() call

/environment/miniconda3/envs/asr/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-26 01:19:45.202262: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-26 01:19:45.283115: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-26 01:19:45.665592: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFl

tf.Tensor(
[[50258 50259 50359 50363  2221    13  2326   388   391   307   264 50244
    295   264  2808  5359    11   293   321   366  5404   281  2928   702
  14943    13 50257]], shape=(1, 27), dtype=int32)
<|startoftranscript|><|en|><|transcribe|><|notimestamps|> Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.<|endoftext|>


## Prompt fix, patch to make forced_decoder_ids work

In [3]:
import tensorflow as tf
import numpy as np
from transformers import TFForceTokensLogitsProcessor, TFLogitsProcessor
from typing import List, Optional, Union, Any

# Patching methods of class TFForceTokensLogitsProcessor(TFLogitsProcessor):

def my__init__(self, force_token_map: List[List[int]]):
    force_token_map = dict(force_token_map)
    # Converts the dictionary of format {index: token} containing the tokens to be forced to an array, where the
    # index of the array corresponds to the index of the token to be forced, for XLA compatibility.
    # Indexes without forced tokens will have an negative value.
    force_token_array = np.ones((max(force_token_map.keys()) + 1), dtype=np.int32) * -1
    for index, token in force_token_map.items():
        if token is not None:
            force_token_array[index] = token
    self.force_token_array = tf.convert_to_tensor(force_token_array, dtype=tf.int32)

def my__call__(self, input_ids: tf.Tensor, scores: tf.Tensor, cur_len: int) -> tf.Tensor:
    def _force_token(generation_idx):
        batch_size = scores.shape[0]
        current_token = self.force_token_array[generation_idx]

        # Original code below generates NaN values when the model is exported to tflite
        # it just needs to be a negative number so that the forced token's value of 0 is the largest
        # so it will get chosen
        #new_scores = tf.ones_like(scores, dtype=scores.dtype) * -float("inf")
        new_scores = tf.ones_like(scores, dtype=scores.dtype) * -float(1)
        indices = tf.stack((tf.range(batch_size), tf.tile([current_token], [batch_size])), axis=1)
        updates = tf.zeros((batch_size,), dtype=scores.dtype)
        new_scores = tf.tensor_scatter_nd_update(new_scores, indices, updates)
        return new_scores

    scores = tf.cond(
        tf.greater_equal(cur_len, tf.shape(self.force_token_array)[0]),
        # If the current length is geq than the length of force_token_array, the processor does nothing.
        lambda: tf.identity(scores),
        # Otherwise, it may force a certain token.
        lambda: tf.cond(
            tf.greater_equal(self.force_token_array[cur_len], 0),
            # Only valid (positive) tokens are forced
            lambda: _force_token(cur_len),
            # Otherwise, the processor does nothing.
            lambda: scores,
        ),
    )
    return scores

TFForceTokensLogitsProcessor.__init__ = my__init__
TFForceTokensLogitsProcessor.__call__ = my__call__

##Define a model with a serving signature and save it in TF SavedModel format.















In [4]:
class GenerateModel(tf.Module):
  def __init__(self, model):
    super(GenerateModel, self).__init__()
    self.model = model

  @tf.function(
    # shouldn't need static batch size, but throws exception without it (needs to be fixed)
    input_signature=[
      tf.TensorSpec((1, 80, 3000), tf.float32, name="input_features"),
    ],
  )
  def serving(self, input_features):
    outputs = self.model.generate(
      input_features,
      # change below if you think your output will be bigger
      # aka if you have bigger transcriptions
      # you can make it 200 for example
      max_new_tokens=448,
      return_dict_in_generate=True,
      forced_decoder_ids=forced_decoder_ids,
    )
    return {"sequences": outputs["sequences"]}

generate_model = GenerateModel(model=model)
tf.saved_model.save(generate_model, saved_model_dir, signatures={"serving_default": generate_model.serving})

2026-02-26 01:21:17.229222: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'tf_whisper_for_conditional_generation/model/decoder/cond/ones/packed/tf_whisper_for_conditional_generation/model/decoder/strided_slice_1' with dtype int32
	 [[{{node tf_whisper_for_conditional_generation/model/decoder/cond/ones/packed/tf_whisper_for_conditional_generation/model/decoder/strided_slice_1}}]]
2026-02-26 01:21:17.283667: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'tf_whisper_for_conditional_generation/model/decoder/cond/add/tf_whisper_for_conditional_generation/model/decoder/strided_slice_2' with dtype int32
	 [[{{node tf_whispe

INFO:tensorflow:Assets written to: tf_whisper-small_saved/assets


INFO:tensorflow:Assets written to: tf_whisper-small_saved/assets


## Convert the model from TF SavedModel format to TF lite

In [5]:
# Convert the model
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.target_spec.supported_ops = [
  tf.lite.OpsSet.TFLITE_BUILTINS, # enable TensorFlow Lite ops.
  tf.lite.OpsSet.SELECT_TF_OPS # enable TensorFlow ops.
]

# Learn about post training quantization
# https://www.tensorflow.org/lite/performance/post_training_quantization

# Dynamic range quantization which reduces the size of the model to 25%
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Float16 quantization reduces the size to 50%
# converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

# Save the model
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

2026-02-26 01:22:39.082187: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'decoder_cond_ones_packed_decoder_strided_slice_1' with dtype int32
	 [[{{node decoder_cond_ones_packed_decoder_strided_slice_1}}]]
2026-02-26 01:22:39.082500: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'decoder_cond_add_decoder_strided_slice_2' with dtype int32
	 [[{{node decoder_cond_add_decoder_strided_slice_2}}]]
2026-02-26 01:22:39.087550: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must fe

##Test tflite model using TFLite Interpreter. Check transcription for dataset



In [6]:
generated_ids = generate_model.serving(input_features)
print(generated_ids)
transcription = processor.batch_decode(generated_ids["sequences"], skip_special_tokens=False)[0]
print(transcription)

2026-02-26 01:24:21.422901: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'tf_whisper_for_conditional_generation/model/decoder/cond/ones/packed/tf_whisper_for_conditional_generation/model/decoder/strided_slice_1' with dtype int32
	 [[{{node tf_whisper_for_conditional_generation/model/decoder/cond/ones/packed/tf_whisper_for_conditional_generation/model/decoder/strided_slice_1}}]]
2026-02-26 01:24:21.430921: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'tf_whisper_for_conditional_generation/model/decoder/cond/add/tf_whisper_for_conditional_generation/model/decoder/strided_slice_2' with dtype int32
	 [[{{node tf_whispe

{'sequences': <tf.Tensor: shape=(1, 449), dtype=int32, numpy=
array([[50258, 50260, 50359, 50363, 10554,    13,  2326,   388,   391,
          307,   264, 50244,   295,   264,  2808,  5359,    11,   293,
          321,   366,  5404,   281,  2928,   702, 14943,    13, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 

In [7]:
# loaded model... now with generate!
interpreter = tf.lite.Interpreter(
    model_path=tflite_model_path,
    #experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
)

tflite_generate = interpreter.get_signature_runner()
generated_ids = tflite_generate(input_features=input_features)
print(generated_ids)

transcription = processor.batch_decode(generated_ids["sequences"], skip_special_tokens=False)[0]
print(transcription)

{'sequences': array([[50258, 50260, 50359, 50363, 10554,    13,  2326,   388,   391,
          307,   264, 50244,   295,   264,  2808,  5359,   293,   321,
          366,  5404,   281,  2928,   702, 14943,    13, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,
        50257, 50257, 50257, 50257, 50257, 50257, 50257, 50257,

## Install faster-whisper for audio processing and testing model

In [ ]:
!git clone https://github.com/SYSTRAN/faster-whisper.git
!pip install faster-whisper

## Test all audio files in loop

In [ ]:
import os
import tensorflow as tf
from transformers import WhisperProcessor, WhisperFeatureExtractor
from faster_whisper import decode_audio

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Set up paths and model (whisper-tiny, whisper-tiny.en, whisper-base, whisper-base.en, whisper-small, whisper-small.en)
# model_name = "whisper-base.en"
# pretrained_model = f"openai/{model_name}"
# tflite_model_path = f"{model_name}.tflite"

audio_path = '/home/featurize/work/whisper_android/whisper_java/app/src/main/assets/english_test1.wav'
tflite_model_path = '/home/featurize/work/whisper_android/models_and_scripts/whisper-tiny.tflite'

feature_extractor = WhisperFeatureExtractor.from_pretrained(pretrained_model)
tokenizer = WhisperTokenizer.from_pretrained(pretrained_model, predict_timestamps=True)
processor = WhisperProcessor(feature_extractor, tokenizer)

interpreter = tf.lite.Interpreter(
    model_path=tflite_model_path,
    experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
)
tflite_generate = interpreter.get_signature_runner()


print(f"Processing {audio_path}...")

# Preprocess the audio file
input_audio = decode_audio(audio_path, sampling_rate=16000)
input_features = feature_extractor(input_audio, sampling_rate=16000, return_tensors="tf").input_features

# Run the model
generated_ids = tflite_generate(input_features=input_features)["sequences"]

# Decode and print transcription
transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(f"{transcription}\n")  # Add newline after each transcription

In [ ]:
import os
import tensorflow as tf
from transformers import WhisperProcessor, WhisperFeatureExtractor
from faster_whisper import decode_audio

# Set up paths and model (whisper-tiny, whisper-tiny.en, whisper-base, whisper-base.en, whisper-small, whisper-small.en)
# model_name = "whisper-base.en"
# pretrained_model = f"openai/{model_name}"
# tflite_model_path = f"{model_name}.tflite"

######## NOTE: Specify the folder containing audio files
!git clone https://github.com/vilassn/audio_samples.git
audio_folder_path = 'audio_samples/en'
#audio_folder_path = '/content/drive/MyDrive/Colab Notebooks/audio'

feature_extractor = WhisperFeatureExtractor.from_pretrained(pretrained_model)
tokenizer = WhisperTokenizer.from_pretrained(pretrained_model, predict_timestamps=True)
processor = WhisperProcessor(feature_extractor, tokenizer)

interpreter = tf.lite.Interpreter(
    model_path=tflite_model_path,
    experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
)
tflite_generate = interpreter.get_signature_runner()

# Number of iterations you want the loop to run
iterations = 1000

for i in range(1, iterations + 1):  # Start from 1 to print iteration number
    print(f"Iteration {i}.......................................................\n")  # Print iteration number and newline

    # Loop through all files in the folder
    for audio_file_name in os.listdir(audio_folder_path):
        audio_file_path = os.path.join(audio_folder_path, audio_file_name)

        if audio_file_name.endswith('.wav'):  # Process only .wav files
            print(f"Processing {audio_file_name}...")

            # Preprocess the audio file
            input_audio = decode_audio(audio_file_path, sampling_rate=16000)
            input_features = feature_extractor(input_audio, sampling_rate=16000, return_tensors="tf").input_features

            # Run the model
            generated_ids = tflite_generate(input_features=input_features)["sequences"]

            # Decode and print transcription
            transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            print(f"{transcription}\n")  # Add newline after each transcription